In [108]:
import pandas as pd
from pathlib import Path

In [109]:
PROCESSED_DATA_DIR = Path("../data/processed")
SCORES_DATA_DIR = Path("../data/scores")
OUTPUTS_DATA_DIR = Path("../data/outputs") 

In [110]:
time_weights = {
    "recent": 0.60,
    "historical": 0.40
}

minimum_coverage_target = 0.90
recommended_coverage_target = 0.99

excluded_statuses = ["Cancelada"]

assert abs(sum(time_weights.values()) - 1) < 1e-9, (
    "Time weights must sum to 1."
)

assert minimum_coverage_target < recommended_coverage_target, (
    "The minimum coverage target must be lower than the recommended target."
)

In [111]:
info_actions_df = pd.read_csv(
    PROCESSED_DATA_DIR / "info_actions.csv",
    sep=",",
    encoding="utf-8"
)

actions_df = pd.read_csv(
    PROCESSED_DATA_DIR / "training_actions.csv",
    sep=",",
    encoding="utf-8",
    parse_dates=[
        "start_date",
        "end_date"
    ]
)

trainers_course_df = pd.read_csv(
    PROCESSED_DATA_DIR / "trainers_course.csv",
    sep=",",
    encoding="utf-8"
)

trainers_local_df = pd.read_csv(
    PROCESSED_DATA_DIR / "trainers_local.csv",
    sep=",",
    encoding="utf-8"
)

priority_df = pd.read_csv(
    OUTPUTS_DATA_DIR / "priority.csv",
    sep=",",
    encoding="utf-8"
)

## Course-local combinations and current trainer capacity

In [112]:
active_courses_df = (
    info_actions_df
    .loc[
        info_actions_df["active"],
        [
            "course_id",
            "course_name",
            "course_area"
        ]
    ]
    .drop_duplicates()
)

locals_df = (
    actions_df[
        ["local"]
    ]
    .drop_duplicates()
    .sort_values("local")
)

course_local_df = (
    active_courses_df
    .merge(
        locals_df,
        how="cross"
    )
)

In [113]:
trainer_eligibility_df = (
    trainers_course_df
    .merge(
        trainers_local_df,
        how="inner",
        on="trainer_id",
        validate="many_to_many"
    )
)

current_trainer_count_df = (
    trainer_eligibility_df
    .groupby(
        [
            "course_id",
            "local"
        ],
        as_index=False
    )
    .agg(
        current_trainer_count=(
            "trainer_id",
            "nunique"
        )
    )
)

course_local_df = (
    course_local_df
    .merge(
        current_trainer_count_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
    .fillna({
        "current_trainer_count": 0
    })
)

course_local_df["current_trainer_count"] = (
    course_local_df["current_trainer_count"]
    .astype(int)
)

course_local_df

,course_id,course_name,course_area,local,current_trainer_count
0,CYB,Cibersegurança Básica,Informática,Centro 1,3
1,CYB,Cibersegurança Básica,Informática,Centro 2,3
2,CYB,Cibersegurança Básica,Informática,Centro 3,4
3,CYB,Cibersegurança Básica,Informática,Centro 4,4
4,CYB,Cibersegurança Básica,Informática,Centro 5,4
...,...,...,...,...,...
115,VND,Vendas e Negociação,Comercial,Centro 1,4
116,VND,Vendas e Negociação,Comercial,Centro 2,4
117,VND,Vendas e Negociação,Comercial,Centro 3,4
118,VND,Vendas e Negociação,Comercial,Centro 4,2


## Daily operational demand

Cancelled actions are excluded. The remaining actions are expanded into one row per active calendar day.

In [114]:
capacity_actions_df = (
    actions_df
    .loc[
        ~actions_df["status"]
        .isin(excluded_statuses)
    ]
    .copy()
)

reference_date = pd.Timestamp("2026-07-29")

recent_start = (
    reference_date
    - pd.DateOffset(months=12)
    + pd.Timedelta(days=1)
)

reference_date, recent_start

(Timestamp('2026-07-29 00:00:00'), Timestamp('2025-07-30 00:00:00'))

In [115]:
capacity_actions_df["date"] = (
    capacity_actions_df
    .apply(
        lambda row:
            pd.date_range(
                start=row["start_date"],
                end=row["end_date"],
                freq="D"
            ),
        axis=1
    )
)

action_days_df = (
    capacity_actions_df[
        [
            "action_id",
            "course_id",
            "local",
            "date"
        ]
    ]
    .explode(
        "date",
        ignore_index=True
    )
)

daily_demand_df = (
    action_days_df
    .groupby(
        [
            "course_id",
            "local",
            "date"
        ],
        as_index=False
    )
    .agg(
        active_actions=(
            "action_id",
            "nunique"
        )
    )
)

daily_demand_df

,course_id,local,date,active_actions
0,CYB,Centro 1,2018-01-10,1
1,CYB,Centro 1,2018-01-11,1
2,CYB,Centro 1,2018-01-12,1
3,CYB,Centro 1,2018-01-13,1
4,CYB,Centro 1,2018-01-14,1
...,...,...,...,...
83269,VND,Centro 5,2026-12-04,2
83270,VND,Centro 5,2026-12-05,2
83271,VND,Centro 5,2026-12-06,2
83272,VND,Centro 5,2026-12-07,2


## Effective shared trainer capacity

When the same trainer can cover several active demands on the same date, their single daily capacity is divided proportionally across those demands.

In [116]:
trainer_daily_demand_df = (
    daily_demand_df
    .merge(
        trainer_eligibility_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="many_to_many"
    )
)

assert trainer_daily_demand_df["trainer_id"].notna().all(), (
    "At least one active course-local combination has no eligible trainer."
)

trainer_daily_demand_df["trainer_total_active_actions"] = (
    trainer_daily_demand_df
    .groupby(
        [
            "trainer_id",
            "date"
        ]
    )["active_actions"]
    .transform("sum")
)

trainer_daily_demand_df["trainer_capacity_share"] = (
    trainer_daily_demand_df["active_actions"]
    .div(
        trainer_daily_demand_df[
            "trainer_total_active_actions"
        ]
    )
)

In [117]:
daily_capacity_df = (
    trainer_daily_demand_df
    .groupby(
        [
            "course_id",
            "local",
            "date"
        ],
        as_index=False
    )
    .agg(
        active_actions=(
            "active_actions",
            "first"
        ),

        effective_trainer_capacity=(
            "trainer_capacity_share",
            "sum"
        ),

        eligible_trainers=(
            "trainer_id",
            "nunique"
        )
    )
    .assign(
        capacity_gap=lambda df:
            df["active_actions"]
            .sub(
                df[
                    "effective_trainer_capacity"
                ]
            )
            .clip(lower=0)
    )
)

capacity_gap_floor = (
    daily_capacity_df["capacity_gap"]
    .astype(int)
)

daily_capacity_df[
    "additional_trainers_needed_day"
] = (
    capacity_gap_floor
    + daily_capacity_df["capacity_gap"]
    .gt(capacity_gap_floor)
    .astype(int)
)

daily_capacity_df["period"] = (
    daily_capacity_df["date"]
    .ge(recent_start)
    .map({
        True: "recent",
        False: "historical"
    })
)

daily_capacity_df

,course_id,local,date,active_actions,effective_trainer_capacity,eligible_trainers,capacity_gap,additional_trainers_needed_day,period
0,CYB,Centro 1,2018-01-10,1,2.333333,3,0.000000,0,historical
1,CYB,Centro 1,2018-01-11,1,2.500000,3,0.000000,0,historical
2,CYB,Centro 1,2018-01-12,1,2.500000,3,0.000000,0,historical
3,CYB,Centro 1,2018-01-13,1,3.000000,3,0.000000,0,historical
4,CYB,Centro 1,2018-01-14,1,3.000000,3,0.000000,0,historical
...,...,...,...,...,...,...,...,...,...
83269,VND,Centro 5,2026-12-04,2,1.333333,2,0.666667,1,recent
83270,VND,Centro 5,2026-12-05,2,1.333333,2,0.666667,1,recent
83271,VND,Centro 5,2026-12-06,2,1.333333,2,0.666667,1,recent
83272,VND,Centro 5,2026-12-07,2,1.333333,2,0.666667,1,recent


## Demand and shortage summary

In [118]:
period_summary_df = (
    daily_capacity_df
    .groupby(
        [
            "course_id",
            "local",
            "period"
        ],
        as_index=False,
        observed=True
    )
    .agg(
        active_days=(
            "date",
            "nunique"
        ),

        average_active_actions=(
            "active_actions",
            "mean"
        ),

        peak_active_actions=(
            "active_actions",
            "max"
        ),

        average_effective_capacity=(
            "effective_trainer_capacity",
            "mean"
        ),

        peak_additional_trainers_needed=(
            "additional_trainers_needed_day",
            "max"
        )
    )
)

period_summary_wide_df = (
    period_summary_df
    .pivot(
        index=[
            "course_id",
            "local"
        ],
        columns="period"
    )
)

period_summary_wide_df.columns = [
    f"{metric}_{period}"
    for metric, period
    in period_summary_wide_df.columns
]

period_summary_wide_df = (
    period_summary_wide_df
    .reset_index()
)

period_summary_wide_df

,course_id,local,active_days_historical,active_days_recent,average_active_actions_historical,average_active_actions_recent,peak_active_actions_historical,peak_active_actions_recent,average_effective_capacity_historical,average_effective_capacity_recent,peak_additional_trainers_needed_historical,peak_additional_trainers_needed_recent
0,CYB,Centro 1,680,288,1.191176,1.538194,4,4,1.769526,1.426552,3,3
1,CYB,Centro 2,873,278,1.324170,1.751799,6,5,2.471581,2.148074,4,4
2,CYB,Centro 3,664,251,1.165663,1.422311,3,5,2.578173,2.312946,2,3
3,CYB,Centro 4,956,313,1.311715,1.664537,4,5,2.785265,2.363156,2,3
4,CYB,Centro 5,567,171,1.231041,1.578947,5,5,3.030309,2.432087,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...
115,VND,Centro 1,466,116,1.126609,1.103448,3,2,2.967675,2.568272,1,0
116,VND,Centro 2,654,202,1.165138,1.485149,4,4,3.069703,2.705457,1,2
117,VND,Centro 3,371,124,1.088949,1.161290,3,3,3.043822,2.478967,0,1
118,VND,Centro 4,551,194,1.230490,1.252577,4,3,1.244406,1.088918,3,2


## Coverage simulation

The simulation tests every feasible number of dedicated additional trainers. The first value reaching each coverage target becomes the corresponding recommendation.

In [119]:
max_additional_trainers = int(
    daily_capacity_df[
        "additional_trainers_needed_day"
    ]
    .max()
)

additional_candidates_df = pd.DataFrame({
    "additional_trainers":
        range(
            max_additional_trainers + 1
        )
})

coverage_simulation_df = (
    daily_capacity_df[
        [
            "course_id",
            "local",
            "period",
            "additional_trainers_needed_day"
        ]
    ]
    .merge(
        additional_candidates_df,
        how="cross"
    )
    .assign(
        covered=lambda df:
            df["additional_trainers"]
            .ge(
                df[
                    "additional_trainers_needed_day"
                ]
            )
    )
    .groupby(
        [
            "course_id",
            "local",
            "additional_trainers",
            "period"
        ],
        as_index=False,
        observed=True
    )
    .agg(
        coverage_rate=(
            "covered",
            "mean"
        )
    )
)

coverage_simulation_df["coverage_rate"] = (
    coverage_simulation_df["coverage_rate"]
    .mul(100)
)

In [120]:
coverage_simulation_wide_df = (
    coverage_simulation_df
    .pivot(
        index=[
            "course_id",
            "local",
            "additional_trainers"
        ],
        columns="period",
        values="coverage_rate"
    )
    .reset_index()
    .rename(
        columns={
            "recent":
                "coverage_rate_recent",

            "historical":
                "coverage_rate_historical"
        }
    )
)

coverage_simulation_wide_df.columns.name = None

for column in [
    "coverage_rate_recent",
    "coverage_rate_historical"
]:
    if column not in coverage_simulation_wide_df:
        coverage_simulation_wide_df[column] = 100.0

coverage_simulation_wide_df[
    [
        "coverage_rate_recent",
        "coverage_rate_historical"
    ]
] = (
    coverage_simulation_wide_df[
        [
            "coverage_rate_recent",
            "coverage_rate_historical"
        ]
    ]
    .fillna(100)
)

coverage_simulation_wide_df[
    "weighted_coverage_rate"
] = (
    coverage_simulation_wide_df[
        "coverage_rate_recent"
    ]
    .mul(
        time_weights["recent"]
    )
    .add(
        coverage_simulation_wide_df[
            "coverage_rate_historical"
        ]
        .mul(
            time_weights["historical"]
        )
    )
)

coverage_simulation_wide_df

,course_id,local,additional_trainers,coverage_rate_historical,coverage_rate_recent,weighted_coverage_rate
0,CYB,Centro 1,0,74.558824,40.972222,54.406863
1,CYB,Centro 1,1,98.088235,87.152778,91.526961
2,CYB,Centro 1,2,99.705882,98.958333,99.257353
3,CYB,Centro 1,3,100.000000,100.000000,100.000000
4,CYB,Centro 1,4,100.000000,100.000000,100.000000
...,...,...,...,...,...,...
835,VND,Centro 5,2,100.000000,100.000000,100.000000
836,VND,Centro 5,3,100.000000,100.000000,100.000000
837,VND,Centro 5,4,100.000000,100.000000,100.000000
838,VND,Centro 5,5,100.000000,100.000000,100.000000


In [121]:
def select_trainer_requirement(
    coverage_df,
    coverage_target,
    output_name
):

    selected_df = (
        coverage_df
        .loc[
            coverage_df[
                "weighted_coverage_rate"
            ]
            .ge(
                coverage_target * 100
            )
        ]
        .sort_values(
            [
                "course_id",
                "local",
                "additional_trainers"
            ]
        )
        .drop_duplicates(
            [
                "course_id",
                "local"
            ]
        )
        [
            [
                "course_id",
                "local",
                "additional_trainers",
                "coverage_rate_recent",
                "coverage_rate_historical",
                "weighted_coverage_rate"
            ]
        ]
    )

    return selected_df.rename(
        columns={
            "additional_trainers":
                f"{output_name}_additional_trainers",

            "coverage_rate_recent":
                f"{output_name}_coverage_rate_recent",

            "coverage_rate_historical":
                f"{output_name}_coverage_rate_historical",

            "weighted_coverage_rate":
                f"{output_name}_weighted_coverage_rate"
        }
    )

In [122]:
current_coverage_df = (
    coverage_simulation_wide_df
    .loc[
        coverage_simulation_wide_df[
            "additional_trainers"
        ]
        .eq(0),
        [
            "course_id",
            "local",
            "coverage_rate_recent",
            "coverage_rate_historical",
            "weighted_coverage_rate"
        ]
    ]
    .rename(
        columns={
            "coverage_rate_recent":
                "current_coverage_rate_recent",

            "coverage_rate_historical":
                "current_coverage_rate_historical",

            "weighted_coverage_rate":
                "current_weighted_coverage_rate"
        }
    )
)

minimum_requirement_df = (
    select_trainer_requirement(
        coverage_simulation_wide_df,
        minimum_coverage_target,
        "minimum"
    )
)

recommended_requirement_df = (
    select_trainer_requirement(
        coverage_simulation_wide_df,
        recommended_coverage_target,
        "recommended"
    )
)

## Final trainer requirements

In [123]:
priority_columns = [
    "course_id",
    "local",
    "priority_score",
    "priority_level",
    "priority_assessment"
]

trainer_requirements_df = (
    course_local_df
    .merge(
        period_summary_wide_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
    .merge(
        current_coverage_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
    .merge(
        minimum_requirement_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
    .merge(
        recommended_requirement_df,
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
    .merge(
        priority_df[
            priority_columns
        ],
        how="left",
        on=[
            "course_id",
            "local"
        ],
        validate="one_to_one"
    )
)

In [124]:
fill_zero_columns = [
    "minimum_additional_trainers",
    "recommended_additional_trainers"
]

trainer_requirements_df[
    fill_zero_columns
] = (
    trainer_requirements_df[
        fill_zero_columns
    ]
    .fillna(0)
    .astype(int)
)

coverage_columns = [
    column
    for column
    in trainer_requirements_df.columns
    if "coverage_rate" in column
]

trainer_requirements_df[
    coverage_columns
] = (
    trainer_requirements_df[
        coverage_columns
    ]
    .fillna(100)
    .round(2)
)

trainer_requirements_df = (
    trainer_requirements_df
    .assign(
        minimum_total_trainers=lambda df:
            df["current_trainer_count"]
            .add(
                df[
                    "minimum_additional_trainers"
                ]
            ),

        recommended_total_trainers=lambda df:
            df["current_trainer_count"]
            .add(
                df[
                    "recommended_additional_trainers"
                ]
            )
    )
)

In [125]:
trainer_requirements_df = (
    trainer_requirements_df[
        [
            "course_id",
            "course_name",
            "course_area",
            "local",

            "priority_score",
            "priority_level",
            "priority_assessment",

            "current_trainer_count",

            "minimum_additional_trainers",
            "recommended_additional_trainers",

            "minimum_total_trainers",
            "recommended_total_trainers",

            "current_weighted_coverage_rate",
            "minimum_weighted_coverage_rate",
            "recommended_weighted_coverage_rate",

            "current_coverage_rate_recent",
            "current_coverage_rate_historical",

            "active_days_recent",
            "active_days_historical",

            "peak_active_actions_recent",
            "peak_active_actions_historical",

            "peak_additional_trainers_needed_recent",
            "peak_additional_trainers_needed_historical"
        ]
    ]
    .sort_values(
        [
            "recommended_additional_trainers",
            "priority_score",
            "current_weighted_coverage_rate"
        ],
        ascending=[
            False,
            False,
            True
        ],
        ignore_index=True
    )
)

trainer_requirements_df

,course_id,course_name,course_area,local,priority_score,priority_level,priority_assessment,current_trainer_count,minimum_additional_trainers,recommended_additional_trainers,...,minimum_weighted_coverage_rate,recommended_weighted_coverage_rate,current_coverage_rate_recent,current_coverage_rate_historical,active_days_recent,active_days_historical,peak_active_actions_recent,peak_active_actions_historical,peak_additional_trainers_needed_recent,peak_additional_trainers_needed_historical
0,LOG,Logística e Gestão de Armazém,Logística,Centro 5,81.44,High,Confirmed high priority,2,1,3,...,93.05,100.00,62.58,84.24,163,495,3,3,3,2
1,PBI,Power BI e Visualização de Dados,Informática,Centro 2,80.72,High,High priority with moderate agreement,4,1,3,...,92.24,99.38,76.29,95.24,388,1072,7,5,4,2
2,PYT,Introdução à Programação em Python,Informática,Centro 2,80.53,High,High priority with moderate agreement,3,1,3,...,91.67,99.63,67.89,87.36,327,957,5,5,4,3
3,FOR,Formação Pedagógica Inicial de Formadores,Formação,Centro 3,77.96,High,High priority with moderate agreement,3,1,3,...,93.43,100.00,70.82,91.03,490,1393,5,5,3,3
4,CYB,Cibersegurança Básica,Informática,Centro 2,77.65,High,High priority with moderate agreement,3,1,3,...,92.47,99.65,73.02,94.50,278,873,5,6,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 3,22.82,Low,Low priority,4,0,0,...,100.00,100.00,100.00,100.00,91,283,3,2,0,0
116,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,22.72,Low,Low priority,2,0,0,...,100.00,100.00,100.00,100.00,75,192,1,2,0,0
117,VND,Vendas e Negociação,Comercial,Centro 3,21.93,Low,Low priority,4,0,0,...,99.52,99.52,99.19,100.00,124,371,3,3,1,0
118,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,21.62,Low,Low priority,4,0,0,...,100.00,100.00,100.00,100.00,70,307,3,2,0,0


## Validation

The recommendation must reach the selected coverage target for every combination represented in the operational history.

In [126]:
expected_combinations = len(
    course_local_df
)

assert len(trainer_requirements_df) == expected_combinations, (
    "Some course-local combinations were lost."
)

assert not trainer_requirements_df.duplicated(
    [
        "course_id",
        "local"
    ]
).any(), (
    "Duplicate course-local combinations found."
)

assert trainer_requirements_df[
    "recommended_additional_trainers"
].ge(
    trainer_requirements_df[
        "minimum_additional_trainers"
    ]
).all(), (
    "Recommended additional trainers cannot be lower than the minimum."
)

assert trainer_requirements_df[
    "recommended_total_trainers"
].ge(
    trainer_requirements_df[
        "current_trainer_count"
    ]
).all(), (
    "Recommended total trainers cannot be lower than the current count."
)

assert trainer_requirements_df[
    "recommended_weighted_coverage_rate"
].ge(
    recommended_coverage_target * 100
).all(), (
    "At least one recommendation does not reach the target coverage."
)

In [127]:
trainer_requirements_df[
    "minimum_additional_trainers"
].value_counts().sort_index().reset_index()

,minimum_additional_trainers,count
0,0,72
1,1,48


In [128]:
trainer_requirements_df[
    "recommended_additional_trainers"
].value_counts().sort_index().reset_index()

,recommended_additional_trainers,count
0,0,24
1,1,51
2,2,31
3,3,14


In [129]:
priority_capacity_summary = (
    trainer_requirements_df
    .groupby(
        "priority_level",
        observed=True
    )
    .agg(
        combinations=(
            "course_id",
            "size"
        ),

        average_priority_score=(
            "priority_score",
            "mean"
        ),

        average_current_coverage=(
            "current_weighted_coverage_rate",
            "mean"
        ),

        average_additional_trainers=(
            "recommended_additional_trainers",
            "mean"
        ),

        median_additional_trainers=(
            "recommended_additional_trainers",
            "median"
        ),

        maximum_additional_trainers=(
            "recommended_additional_trainers",
            "max"
        )
    )
    .round(2)
)

priority_capacity_summary

,combinations,average_priority_score,average_current_coverage,average_additional_trainers,median_additional_trainers,maximum_additional_trainers
priority_level,,,,,,
High,11,79.87,76.45,2.55,3.0,3
Low,10,21.65,94.97,0.30,0.0,2
Lower-middle,52,39.13,95.44,0.81,1.0,3
Upper-middle,47,61.93,85.86,1.74,2.0,3


In [130]:
priority_capacity_correlation = (
    trainer_requirements_df["priority_score"]
    .rank(method="average")
    .corr(
        trainer_requirements_df[
            "recommended_additional_trainers"
        ]
        .rank(method="average")
    )
)

print(priority_capacity_correlation.round(3))

0.777


In [131]:
priority_coverage_correlation = (
    trainer_requirements_df["priority_score"]
    .rank(method="average")
    .corr(
        trainer_requirements_df[
            "current_weighted_coverage_rate"
        ]
        .rank(method="average")
    )
)

print(priority_coverage_correlation.round(3))

-0.751


In [132]:
trainer_requirements_df = trainer_requirements_df[
    [
        'course_id',
        'course_name',
        'course_area',
        'local',

        'current_trainer_count',
        'minimum_additional_trainers',
        'recommended_additional_trainers',

        'minimum_total_trainers',
        'recommended_total_trainers',

        'current_weighted_coverage_rate',
        'minimum_weighted_coverage_rate',
        'recommended_weighted_coverage_rate',

        'current_coverage_rate_recent',
        'current_coverage_rate_historical',

        'active_days_recent',
        'active_days_historical',

        'peak_active_actions_recent',
        'peak_active_actions_historical',

        'peak_additional_trainers_needed_recent',
        'peak_additional_trainers_needed_historical'
    ]
]

trainer_requirements_df

,course_id,course_name,course_area,local,current_trainer_count,minimum_additional_trainers,recommended_additional_trainers,minimum_total_trainers,recommended_total_trainers,current_weighted_coverage_rate,minimum_weighted_coverage_rate,recommended_weighted_coverage_rate,current_coverage_rate_recent,current_coverage_rate_historical,active_days_recent,active_days_historical,peak_active_actions_recent,peak_active_actions_historical,peak_additional_trainers_needed_recent,peak_additional_trainers_needed_historical
0,LOG,Logística e Gestão de Armazém,Logística,Centro 5,2,1,3,3,5,71.24,93.05,100.00,62.58,84.24,163,495,3,3,3,2
1,PBI,Power BI e Visualização de Dados,Informática,Centro 2,4,1,3,5,7,83.87,92.24,99.38,76.29,95.24,388,1072,7,5,4,2
2,PYT,Introdução à Programação em Python,Informática,Centro 2,3,1,3,4,6,75.68,91.67,99.63,67.89,87.36,327,957,5,5,4,3
3,FOR,Formação Pedagógica Inicial de Formadores,Formação,Centro 3,3,1,3,4,6,78.90,93.43,100.00,70.82,91.03,490,1393,5,5,3,3
4,CYB,Cibersegurança Básica,Informática,Centro 2,3,1,3,4,6,81.61,92.47,99.65,73.02,94.50,278,873,5,6,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 3,4,0,0,4,4,100.00,100.00,100.00,100.00,100.00,91,283,3,2,0,0
116,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,2,0,0,2,2,100.00,100.00,100.00,100.00,100.00,75,192,1,2,0,0
117,VND,Vendas e Negociação,Comercial,Centro 3,4,0,0,4,4,99.52,99.52,99.52,99.19,100.00,124,371,3,3,1,0
118,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,4,0,0,4,4,100.00,100.00,100.00,100.00,100.00,70,307,3,2,0,0


## Save results

In [133]:
trainer_requirements_df.to_csv(
    OUTPUTS_DATA_DIR / "trainer_capacity_requirements.csv",
    index=False,
    encoding="utf-8"
)